# Giai đoạn 3 — Bước 8A: Inference Preflight (chỉ nghiên cứu)

**Bạn tự chạy notebook.** Chỉ dùng pipeline LogisticRegression từ Bước 5 với cùng môi trường; **KHÔNG train**, KHÔNG mở Test, KHÔNG đổi ngưỡng 0,225, KHÔNG dùng dữ liệu sức khỏe người thật. Ở Bước 8A, chỉ thực hành API-style inference cục bộ, chưa mở máy chủ FastAPI.

**Trình tự:** (1) kiểm tra file/môi trường/hash, (2) nạp artifact đáng tin, (3) tái kiểm tra Validation, (4) validate input giả lập, (5) local inference, (6) xuất manifest và log. Đọc cảnh báo ở ô nạp mô hình trước khi thực thi.

## 1. Imports và xác định thư mục dự án

Giữ nguyên các tệp do chính bạn tạo: `models/baseline/nhanes_hba1c_baseline_step5.joblib`, `data/outputs/model_training/step5_metadata.json`, `validation.csv`. Không gửi model `.joblib` nếu không cần.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.metrics import roc_auc_score
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'outputs' / 'model_training').is_dir():
    ROOT = cwd
elif (cwd.parent / 'data' / 'outputs' / 'model_training').is_dir():
    ROOT = cwd.parent
else:
    raise FileNotFoundError('Hãy mở Jupyter từ 03-ML-Research hoặc 03-ML-Research/notebooks.')

MODEL_FILE = ROOT / 'models' / 'baseline' / 'nhanes_hba1c_baseline_step5.joblib'
META_FILE = ROOT / 'data' / 'outputs' / 'model_training' / 'step5_metadata.json'
VAL_FILE = ROOT / 'data' / 'outputs' / 'model_training' / 'validation.csv'
OUT_DIR = ROOT / 'data' / 'outputs' / 'model_deployment' / 'preflight_8a'

for p in (MODEL_FILE, META_FILE, VAL_FILE):
    if not p.is_file():
        raise FileNotFoundError(f'Thiếu file: {p}')
print('Python:', platform.python_version(), '| sklearn:', sklearn.__version__, '| pandas:', pd.__version__)
print('Model path exists:', MODEL_FILE.exists())
print('CHỈ đọc Validation; KHÔNG đọc test_holdout_DO_NOT_TUNE.csv.')

Python: 3.13.9 | sklearn: 1.8.0 | pandas: 3.0.0
Model path exists: True
CHỈ đọc Validation; KHÔNG đọc test_holdout_DO_NOT_TUNE.csv.


## 2. Kiểm tra schema / phiên bản / SHA256 **trước khi nạp model**

`joblib.load()` dùng pickle và **có thể chạy mã độc khi file không đáng tin**. Chỉ nạp mô hình do chính bạn tạo ở Bước 5, không nhận `.joblib` từ người lạ. scikit-learn không hỗ trợ tin cậy việc nạp model bằng phiên bản khác lúc train.

In [2]:
EXPECTED_SHA256 = '31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1'
FEATURES = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI']
TARGET = 'a1c_ge_57_candidate'
LOCKED_THRESHOLD = 0.225
EXPECTED_N_VAL = 705
EXPECTED_N_POS = 208

metadata = json.loads(META_FILE.read_text(encoding='utf-8'))
computed_sha = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()
assert computed_sha == EXPECTED_SHA256, 'SHA256 không khớp! KHÔNG nạp model; kiểm tra artifact gốc.'
assert metadata.get('features') == FEATURES, 'Schema khác Bước 5.'
assert metadata.get('selected_model') == 'LogisticRegression', 'Sai mô hình đã khóa.'
assert metadata.get('sklearn_version') == sklearn.__version__, 'Phiên bản sklearn khác lần train.'
assert metadata.get('python_version') == platform.python_version(), 'Python khác lần train.'
print('SHA256 confirmed:',computed_sha)
print('Metadata model:',metadata.get('selected_model'))
print('Schema:',FEATURES)
print('Threshold locked:',LOCKED_THRESHOLD)
print('OK: có thể chuyển sang ô nạp model đáng tin.')

SHA256 confirmed: 31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1
Metadata model: LogisticRegression
Schema: ['RIDAGEYR', 'RIAGENDR', 'BMXBMI']
Threshold locked: 0.225
OK: có thể chuyển sang ô nạp model đáng tin.


## 3. **CẢNH BÁO**: Nạp mô hình `joblib` đáng tin từ máy bạn

Chỉ chạy sau khi ô trước **thành công**. Không chạy khi checksum hoặc thư viện không khớp. Không dùng file do người khác cung cấp. Notebook **không huấn luyện**.

In [3]:
pipe = joblib.load(MODEL_FILE)  # CHỈ model do chính bạn huấn luyện và đã kiểm SHA256
assert list(pipe.named_steps) == ['prep', 'model'], 'Pipeline không đúng cấu trúc cũ.'
assert type(pipe.named_steps['model']).__name__ == 'LogisticRegression', 'Sai estimator.'
print('Đã nạp pipeline có sẵn; KHÔNG có fit() trong notebook.')

Đã nạp pipeline có sẵn; KHÔNG có fit() trong notebook.


## 4. Tái xác minh đầu ra Validation (KHÔNG dùng Test)

Điều này chỉ bảo đảm bản đóng gói đang dùng đúng pipeline Bước 5, **không phải đánh giá lại khả năng tổng quát hóa**. Test ở Bước 6B đã khóa và không được mở lại.

In [4]:
val = pd.read_csv(VAL_FILE)
assert set(FEATURES + [TARGET]).issubset(val.columns), 'Validation không đúng schema.'
assert len(val) == EXPECTED_N_VAL, 'Validation count sai.'
assert int(val[TARGET].sum()) == EXPECTED_N_POS, 'Positive n sai.'
p_val = pipe.predict_proba(val[FEATURES])[:, 1]
auc_val = float(roc_auc_score(val[TARGET].astype(int), p_val))
assert abs(auc_val - 0.7173) < 0.001, f'Validation AUC khác: {auc_val}'
assert np.isfinite(p_val).all() and np.logical_and(p_val >= 0, p_val <= 1).all()
print('Validation:',len(val),'| Positive:',int(val[TARGET].sum()),'| ROC-AUC reproduced:',round(auc_val,4))
print('TEST OPENED: NO | FIT: NO')

Validation: 705 | Positive: 208 | ROC-AUC reproduced: 0.7173
TEST OPENED: NO | FIT: NO


## 5. Hợp đồng dữ liệu và kiểm tra đầu vào

**Chỉ dữ liệu giả lập**. Giới hạn bên dưới là giới hạn *research preflight* theo phạm vi từng quan sát trong bộ dữ liệu thực hành, không phải ngưỡng y khoa. `RIDAGEYR=80` có nghĩa **80 tuổi trở lên** trong NHANES. Để tránh giả định ngoài phạm vi đã học, trong 8A từ chối tuổi >80 và BMI ngoài 11,1..74,8. API trong tương lai có thể cần thiết kế chính sách dữ liệu thiếu khác, không tự thay bằng 0.

In [5]:
def validate_research_input(payload: dict) -> dict:
    if not isinstance(payload, dict):
        raise ValueError('Input phải là một JSON object/dict.')
    if set(payload) != set(FEATURES):
        raise ValueError(f'Phải có đúng 3 trường {FEATURES}, không thêm PII hoặc biến khác.')
    age, gender, bmi = (payload[k] for k in FEATURES)
    if isinstance(age, (bool, np.bool_)) or not isinstance(age, (int, np.integer)) or not 20 <= age <= 80:
        raise ValueError('RIDAGEYR phải là số nguyên trong khoảng 20..80 (80 = nhóm 80+).')
    if isinstance(gender, (bool, np.bool_)) or not isinstance(gender, (int, np.integer)) or gender not in (1, 2):
        raise ValueError('RIAGENDR phải là mã 1 hoặc 2 của khảo sát NHANES.')
    if isinstance(bmi, (bool, np.bool_)) or not isinstance(bmi, (int, float, np.integer, np.floating)):
        raise ValueError('BMXBMI phải là số hữu hạn.')
    if not math.isfinite(float(bmi)) or not (11.1 <= float(bmi) <= 74.8):
        raise ValueError('BMXBMI nằm ngoài phạm vi thực nghiệm (11.1..74.8 kg/m²).')
    return {'RIDAGEYR': int(age), 'RIAGENDR': int(gender), 'BMXBMI': float(bmi)}

examples = [
    {'RIDAGEYR': 45, 'RIAGENDR': 1, 'BMXBMI': 28.5},
    {'RIDAGEYR': 60, 'RIAGENDR': 2, 'BMXBMI': 34.8},
]
validated_examples = [validate_research_input(x) for x in examples]
for bad in [
    {'RIDAGEYR': 19, 'RIAGENDR': 1, 'BMXBMI': 28.5},
    {'RIDAGEYR': 45, 'RIAGENDR': 3, 'BMXBMI': 28.5},
    {'RIDAGEYR': 45, 'RIAGENDR': 1, 'BMXBMI': float('nan')},
    {'RIDAGEYR': 45, 'RIAGENDR': 1},
]:
    try:
        validate_research_input(bad)
        raise AssertionError('Input không hợp lệ đã lọt qua!')
    except ValueError:
        pass
print('Input validation tests: PASSED (4 trường hợp lỗi).')

Input validation tests: PASSED (4 trường hợp lỗi).


## 6. Inference cục bộ — dữ liệu **giả lập**, KHÔNG phải dự đoán bệnh

Chỉ đọc pipeline đã fit và gọi `.predict_proba` cho 2 ví dụ giả lập, giữ threshold 0,225. Dùng `model_score`, không dùng `diabetes_probability`. Đừng đưa điểm số này như khuyến nghị y khoa cho bất kỳ người nào.

In [6]:
X_demo = pd.DataFrame(validated_examples, columns=FEATURES)
scores = pipe.predict_proba(X_demo)[:, 1]
assert np.isfinite(scores).all() and np.logical_and(scores >= 0, scores <= 1).all()
results_demo = []
for i, score in enumerate(scores, start=1):
    results_demo.append({
        'synthetic_case': i,
        'status': 'research_only',
        'task': 'classify_hba1c_ge_5_7_indicator',
        'model_name': 'LogisticRegression',
        'threshold': LOCKED_THRESHOLD,
        'model_score': float(score),
        'predicted_label': int(score >= LOCKED_THRESHOLD),
        'warning': 'Research-only classification of observed HbA1c indicator; not a diagnosis or future type 2 diabetes risk estimate.'
    })
display(pd.DataFrame(results_demo)[['synthetic_case','model_score','predicted_label','threshold']].round(5))
assert np.array_equal(scores, pipe.predict_proba(X_demo)[:, 1]), 'Same input must give same score.'
print('Synthetic inference successful; model unchanged.')

,synthetic_case,model_score,predicted_label,threshold
0,1,0.22552,1,0.225
1,2,0.39459,1,0.225


Synthetic inference successful; model unchanged.


## 7. Xuất manifest kỹ thuật (không chứa dữ liệu cá nhân)

Lưu metadata để sau này phát triển FastAPI (Bước 8B). Không copy hoặc sửa file model gốc, không gửi tệp model cho người khác. `manifest` không có thông tin sức khỏe cá thể.

In [7]:
manifest = {
    'experiment': 'EXP-DEPLOY-8A-001',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'stage': 'phase3_step8a_local_inference_preflight',
    'status': 'research_only_not_clinically_validated',
    'source': 'NHANES 2021-2023 subset, screening_candidate',
    'model': 'LogisticRegression',
    'model_sha256': computed_sha,
    'python_version': platform.python_version(),
    'sklearn_version': sklearn.__version__,
    'features': FEATURES,
    'feature_schema': {
        'RIDAGEYR': 'integer 20..80; 80 is top-coded 80+',
        'RIAGENDR': 'integer code 1 or 2',
        'BMXBMI': 'finite float 11.1..74.8 kg/m2 (research domain, not clinical limits)'
    },
    'label': 'HbA1c >= 5.7%, not a type 2 diagnosis',
    'locked_threshold': LOCKED_THRESHOLD,
    'validation_roc_auc_reproduced': auc_val,
    'validation_n': int(len(val)),
    'validation_positive_n': int(val[TARGET].sum()),
    'test_reopened': False,
    'fit_executed': False,
    'model_modified': False,
    'synthetic_input_tests_n': len(results_demo),
    'synthetic_invalid_cases_rejected': 4,
    'limitations': [
        'NOT a clinical diagnostic tool or future type 2 diabetes risk score',
        'not externally/clinically validated',
        'NHANES complex-survey weights/design not accounted for in training/evaluation',
        'pregnancy status incomplete',
        'positive alerts may be frequent',
        'no model tuning or repeat test evaluation after holdout was opened'
    ]
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_FILE = OUT_DIR / '08a_inference_manifest.json'
MANIFEST_FILE.write_text(json.dumps(manifest,ensure_ascii=False,indent=2), encoding='utf-8')
print('Manifest saved:', MANIFEST_FILE)
print('No individual data saved in manifest.')

Manifest saved: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_deployment\preflight_8a\08a_inference_manifest.json
No individual data saved in manifest.


## 8. Output cuối cần gửi lại

Gửi phần output cuối bên dưới và (nếu tiện) `08a_inference_manifest.json`. Không gửi `.joblib` và không gửi thông tin sức khỏe của người thật. **Chỉ khi 8A đạt yêu cầu, chúng ta mới triển khai 8B FastAPI theo từng bước do bạn tự thực hiện.**

In [8]:
print('===== BƯỚC 8A — INFERENCE PREFLIGHT (NGƯỜI DÙNG TỰ CHẠY) =====')
print('Model:', manifest['model'])
print('SHA256:', computed_sha)
print('Python:', platform.python_version(), '| sklearn:', sklearn.__version__)
print('Validation:',len(val),'| Positive:',int(val[TARGET].sum()),'| ROC-AUC:',round(auc_val,4))
print('FEATURES:',FEATURES,'| LOCKED THRESHOLD:',LOCKED_THRESHOLD)
print('---- SYNTHETIC LOCAL INFERENCE ----')
print(pd.DataFrame(results_demo)[['synthetic_case','model_score','predicted_label']].round(5).to_string(index=False))
print('Input rejection tests: 4/4 passed')
print('Saved manifest:', MANIFEST_FILE)
print('TRAIN FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | API SERVER STARTED: NO')
print('===== KẾT THÚC — GỬI OUTPUT CHO TÔI TRƯỚC BƯỚC 8B =====')

===== BƯỚC 8A — INFERENCE PREFLIGHT (NGƯỜI DÙNG TỰ CHẠY) =====
Model: LogisticRegression
SHA256: 31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1
Python: 3.13.9 | sklearn: 1.8.0
Validation: 705 | Positive: 208 | ROC-AUC: 0.7173
FEATURES: ['RIDAGEYR', 'RIAGENDR', 'BMXBMI'] | LOCKED THRESHOLD: 0.225
---- SYNTHETIC LOCAL INFERENCE ----
 synthetic_case  model_score  predicted_label
              1      0.22552                1
              2      0.39459                1
Input rejection tests: 4/4 passed
Saved manifest: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_deployment\preflight_8a\08a_inference_manifest.json
TRAIN FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | API SERVER STARTED: NO
===== KẾT THÚC — GỬI OUTPUT CHO TÔI TRƯỚC BƯỚC 8B =====
